In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [35]:
df = pd.read_csv("adult11.csv")

In [36]:
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,salary
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       48842 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      48842 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   gender          48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48842 non-null  object
 14  salary          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


In [7]:
df.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


In [10]:
df.isnull().sum()

,0
age,0
workclass,0
fnlwgt,0
education,0
education-num,0
marital-status,0
occupation,0
relationship,0
race,0
gender,0


Feature Engineering

In [16]:
df['Age_Group'] = pd.cut(df['age'],
                         bins=[0,25,45,65,100],
                         labels=[0,1,2,3])

In [17]:
df['Work_Intensity'] = (
    df['hours-per-week']
    * df['education-num']
)

In [18]:
df['Total_Capital'] = (
    df['capital-gain']
    - df['capital-loss']
)

Encoding

In [19]:
le = LabelEncoder()

In [20]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

In [23]:
X = df.drop('salary', axis=1)
y = df['salary']

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [25]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [26]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import *

dt = DecisionTreeClassifier(
    random_state=42
)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print("Decision Tree")
print("Accuracy:",
      accuracy_score(y_test,y_pred_dt))

print(classification_report(
    y_test,
    y_pred_dt
))

print(confusion_matrix(
    y_test,
    y_pred_dt
))

Decision Tree
Accuracy: 0.8165625959668339
              precision    recall  f1-score   support

           0       0.88      0.87      0.88      7479
           1       0.60      0.63      0.62      2290

    accuracy                           0.82      9769
   macro avg       0.74      0.75      0.75      9769
weighted avg       0.82      0.82      0.82      9769

[[6538  941]
 [ 851 1439]]


In [27]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest")
print("Accuracy:",
      accuracy_score(y_test,y_pred_rf))

print(classification_report(
    y_test,
    y_pred_rf
))

print(confusion_matrix(
    y_test,
    y_pred_rf
))

Random Forest
Accuracy: 0.8626266762206981
              precision    recall  f1-score   support

           0       0.89      0.93      0.91      7479
           1       0.74      0.64      0.69      2290

    accuracy                           0.86      9769
   macro avg       0.82      0.79      0.80      9769
weighted avg       0.86      0.86      0.86      9769

[[6956  523]
 [ 819 1471]]


In [28]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

print("Naive Bayes")
print("Accuracy:",
      accuracy_score(y_test,y_pred_nb))

print(classification_report(
    y_test,
    y_pred_nb
))

print(confusion_matrix(
    y_test,
    y_pred_nb
))

Naive Bayes
Accuracy: 0.7992629747159382
              precision    recall  f1-score   support

           0       0.82      0.95      0.88      7479
           1       0.65      0.31      0.42      2290

    accuracy                           0.80      9769
   macro avg       0.73      0.63      0.65      9769
weighted avg       0.78      0.80      0.77      9769

[[7095  384]
 [1577  713]]


In [29]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(
    n_neighbors=5
)

knn.fit(
    X_train_scaled,
    y_train
)

y_pred_knn = knn.predict(
    X_test_scaled
)

print("KNN")

print("Accuracy:",
      accuracy_score(
          y_test,
          y_pred_knn
      ))

print(classification_report(
    y_test,
    y_pred_knn
))

print(confusion_matrix(
    y_test,
    y_pred_knn
))

KNN
Accuracy: 0.8375473436380387
              precision    recall  f1-score   support

           0       0.89      0.90      0.90      7479
           1       0.67      0.62      0.64      2290

    accuracy                           0.84      9769
   macro avg       0.78      0.76      0.77      9769
weighted avg       0.83      0.84      0.84      9769

[[6768  711]
 [ 876 1414]]


This project successfully implemented and evaluated four machine learning classification algorithms on the Adult Income Dataset. The results demonstrated that Random Forest outperformed all other models, achieving an accuracy of 86.26%, precision of 0.74, recall of 0.64, and F1-score of 0.69. KNN and Decision Tree provided acceptable performance, while Naive Bayes produced the lowest results due to its feature independence assumption. Based on the experimental findings, Random Forest is recommended as the most suitable model for Adult Income Classification because of its high accuracy, strong generalization capability, and reliable predictive performance.